In [1]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# ==========================================================
# 1. MOTOR DE NAVEGACIÓN (Configuración de Laboratorio)
# ==========================================================
def iniciar_navegador():
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64)")
    
    return webdriver.Chrome(options=options)

# ==========================================================
# 2. CONFIGURACIÓN DEL GRUPO
# ==========================================================
URL_OBJETIVO = "https://books.toscrape.com/"
PAGINAS_OBJETIVO = 5  # Número de páginas a recorrer

SELECTOR_CONTENEDOR = "article.product_pod"
SELECTOR_DATO_A     = "h3 a"
SELECTOR_DATO_B     = "p.price_color"

# ==========================================================
# 3. EJECUCIÓN DEL SCRAPING DINÁMICO
# ==========================================================
driver = iniciar_navegador()
dataset_final = []

try:
    print(" Conectando a la fuente de datos...")
    driver.get(URL_OBJETIVO)

    # --- BUCLE DE PAGINACIÓN ---
    for i in range(PAGINAS_OBJETIVO):
        print(f"--- Procesando Página {i + 1} ---")

        # Espera explícita para carga
        WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located((By.CSS_SELECTOR, SELECTOR_CONTENEDOR))
        )

        # Captura de elementos
        elementos = driver.find_elements(By.CSS_SELECTOR, SELECTOR_CONTENEDOR)
        print(f" Se encontraron {len(elementos)} registros en esta página.")

        for item in elementos:
            try:
                # Extracción limpia (como en tu primer código)
                columna_a = item.find_element(By.CSS_SELECTOR, SELECTOR_DATO_A).text
                columna_b = item.find_element(By.CSS_SELECTOR, SELECTOR_DATO_B).text

                # Limpieza de datos
                valor_limpio = "".join(filter(str.isdigit, columna_b))

                dataset_final.append({
                    "variable_nombre": columna_a,
                    "variable_valor": valor_limpio,
                    "status": 1.0
                })

            except:
                continue

        # --- NAVEGACIÓN A SIGUIENTE PÁGINA ---
        try:
            boton_siguiente = driver.find_element(By.XPATH, "//a[text()='next']")
            boton_siguiente.click()
            time.sleep(3)

        except:
            print(" No hay más páginas disponibles.")
            break

    # ==========================================================
    # 4. SALIDA DE DATOS
    # ==========================================================
    if dataset_final:
        df = pd.DataFrame(dataset_final)
        nombre_archivo = "datos_dinamicos_grupo.csv"
        df.to_csv(nombre_archivo, index=False)

        print("\n Proceso exitoso.")
        print(f" Total de registros capturados: {len(df)}")
        print(f" Archivo generado: {nombre_archivo}")
        print(df.head())

    else:
        print(" No se capturaron datos.")

except Exception as e:
    print(f" Error durante la ejecución: {e}")

finally:
    driver.quit()
    print(" Navegador cerrado.")

 Conectando a la fuente de datos...
--- Procesando Página 1 ---
 Se encontraron 20 registros en esta página.
--- Procesando Página 2 ---
 Se encontraron 20 registros en esta página.
--- Procesando Página 3 ---
 Se encontraron 20 registros en esta página.
--- Procesando Página 4 ---
 Se encontraron 20 registros en esta página.
--- Procesando Página 5 ---
 Se encontraron 20 registros en esta página.

 Proceso exitoso.
 Total de registros capturados: 100
 Archivo generado: datos_dinamicos_grupo.csv
                variable_nombre variable_valor  status
0            A Light in the ...           5177     1.0
1            Tipping the Velvet           5374     1.0
2                    Soumission           5010     1.0
3                 Sharp Objects           4782     1.0
4  Sapiens: A Brief History ...           5423     1.0
 Navegador cerrado.
